# Antutor 백엔드용 vLLM 서버 (Qwen3-8B-AWQ)

이 노트북은 Google Colab의 **T4 GPU** 환경에서 실행되도록 구성되었습니다.
Qwen3 모델을 하나로 통합하기 위해 vLLM 기반의 OpenAI API 호환 서버를 띄우고, Ngrok을 통해 외부에서 접속할 수 있는 엔드포인트를 발급합니다.

**준비 사항:**
- `런타임` -> `런타임 유형 변경` -> **T4 GPU**가 선택되어 있는지 확인하세요.
- [ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)에서 토큰을 복사해 두세요.

In [ ]:
# 1. 패키지 설치
!pip install vllm pyngrok

In [ ]:
# 2. Ngrok 터널 오픈 및 엔드포인트 확인
import os
from pyngrok import ngrok, conf

# 여기에 본인의 Ngrok 토큰을 입력하세요
NGROK_AUTH_TOKEN = "여기에_토큰_입력"
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# 실행 중인 기존 터널 종료
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# vLLM 서버가 돌아갈 8000번 포트 연결
public_url = ngrok.connect(8000).public_url
print("=" * 70)
print(f"🔥 vLLM API 서버 엔드포인트: {public_url}")
print("💡 위 주소를 복사하여 백엔드 .env 파일의 DRAFT_LLM_ENDPOINT와 DEBATE_LLM_ENDPOINT에 붙여넣으세요!")
print("=" * 70)

In [ ]:
# 3. Qwen3-8B-AWQ vLLM 서버 구동
# T4 VRAM(16GB)에 맞추어 Context 길이를 8192로 제한하고, AWQ 양자화 옵션을 사용합니다.
!python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen3-8B-AWQ \
    --quantization awq \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.9 \
    --port 8000